# Transcribe raw audios with the Faster Whisper API

Sends every file in `data/raw_audios/` to `/jobs`, polls until done, and writes `data/transcripts/<name>.json` (full server response + text) and `<name>.txt`.
Resumable: files that already have a `.json` are skipped. Uploads are streamed from disk (files are hundreds of MB). Stdlib only.

In [1]:
import json, mimetypes, os, sys, time, uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from urllib import error, parse, request

API = "http://192.168.1.71:8199"

# Walk up from the notebook's cwd to the project root (folder with pyproject.toml).
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").exists())

INPUT_DIR  = ROOT / "data" / "raw_audios"
OUTPUT_DIR = ROOT / "data" / "transcripts"

LANGUAGE        = "auto"   # "auto" | "uz" | "kk" | "ru"
WORD_PROBS      = False    # include per-word timings/probabilities in the response
MIN_PROBABILITY = 0.0
WORKERS         = 6        # jobs in flight; server decodes 4 at once, extras keep its queue full
POLL_SECONDS    = 5.0
JOB_TIMEOUT     = 6 * 3600 # max seconds per job (includes time waiting in the server queue)
LOG_EVERY       = 30       # seconds between progress lines while a job is waiting

AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".opus"}
DONE   = {"done", "completed", "complete", "succeeded", "success", "finished"}
FAILED = {"failed", "error", "errored", "cancelled", "canceled"}
IN_PROGRESS = {"queued", "pending", "running", "processing", "started", "in_progress"}
print("ROOT:", ROOT)

ROOT: /Users/macbook/Documents/IT/projects/tts


In [2]:
def http_json(url, *, data=None, headers=None, timeout=60):
    req = request.Request(url, data=data, headers=headers or {})
    try:
        with request.urlopen(req, timeout=timeout) as resp:
            return resp.status, json.loads(resp.read() or b"{}")
    except error.HTTPError as e:
        body = e.read().decode("utf-8", "replace")
        try:
            return e.code, json.loads(body)
        except json.JSONDecodeError:
            return e.code, {"detail": body}


def multipart_stream(fields, file_field, path, chunk=1 << 20):
    """Return (body_generator, content_type, content_length) without loading the file into memory."""
    boundary = uuid.uuid4().hex
    head = b"".join(f'--{boundary}\r\nContent-Disposition: form-data; name="{k}"\r\n\r\n{v}\r\n'.encode()
                    for k, v in fields.items())
    ctype = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    # Header-safe filename (titles contain quotes/unicode); the server only needs the extension.
    head += (f'--{boundary}\r\nContent-Disposition: form-data; name="{file_field}"; '
             f'filename="audio{path.suffix.lower()}"\r\nContent-Type: {ctype}\r\n\r\n').encode()
    tail = f"\r\n--{boundary}--\r\n".encode()
    length = len(head) + path.stat().st_size + len(tail)

    def gen():
        yield head
        with path.open("rb") as f:
            while block := f.read(chunk):
                yield block
        yield tail

    return gen(), f"multipart/form-data; boundary={boundary}", length


def retry(fn, attempts, what):
    delay = 2.0
    for i in range(1, attempts + 1):
        try:
            return fn()
        except (error.URLError, TimeoutError, ConnectionError, RuntimeError) as e:
            if i == attempts:
                raise RuntimeError(f"{what} failed after {attempts} attempts: {e}") from e
            time.sleep(delay)
            delay = min(delay * 2, 30)


def submit(path):
    def go():
        body, ctype, length = multipart_stream(
            {"language": LANGUAGE, "word_probs": str(WORD_PROBS).lower(), "min_probability": str(MIN_PROBABILITY)},
            "file", path)  # fresh generator per attempt
        status, js = http_json(f"{API}/jobs", data=body,
                               headers={"Content-Type": ctype, "Content-Length": str(length)}, timeout=1800)
        if status >= 500:
            raise RuntimeError(f"submit HTTP {status}: {js}")      # retryable
        if status not in (200, 201, 202):
            raise ValueError(f"submit HTTP {status}: {js}")        # not retryable (e.g. 422)
        job_id = js.get("job_id") or js.get("id")
        if not job_id:
            raise ValueError(f"no job id in submit response: {js}")
        return str(job_id)

    return retry(go, 5, "submit")


def wait(job_id, label=""):
    t0 = time.monotonic()
    deadline = t0 + JOB_TIMEOUT
    warned = set()
    last_state, last_log = None, 0.0
    while time.monotonic() < deadline:
        status, js = retry(lambda: http_json(f"{API}/jobs/{parse.quote(job_id)}", timeout=30), 5, "poll")
        if status == 404:
            raise ValueError(f"job {job_id} not found on server (restarted or expired?)")
        if status >= 500:
            time.sleep(POLL_SECONDS)
            continue
        state = str(js.get("status", "")).lower()
        now = time.monotonic()
        if state != last_state or now - last_log >= LOG_EVERY:  # show progress, otherwise a busy server looks like a hang
            try:
                q = http_json(f"{API}/health", timeout=10)[1].get("queue_size")
            except Exception:
                q = "?"
            print(f"  [{label or job_id[:8]}] {state or '?'}  server queue={q}  waited {now - t0:.0f}s")
            last_state, last_log = state, now
        if state in FAILED:
            raise ValueError(f"job {job_id} failed: {js}")
        if state in DONE:
            return js
        if state not in IN_PROGRESS:
            if not state and ("result" in js or "text" in js or "segments" in js):
                return js  # no status field, but the payload is there
            if state not in warned:
                warned.add(state)
                print(f"[warn] unrecognised job status {state!r}: {js}", file=sys.stderr)
        time.sleep(POLL_SECONDS)
    raise TimeoutError(f"job {job_id} not finished after {JOB_TIMEOUT}s")


def extract_text(js):
    res = js.get("result", js)
    if isinstance(res, dict):
        if isinstance(res.get("text"), str):
            return res["text"].strip()
        segs = res.get("segments")
        if isinstance(segs, list):
            return " ".join(s.get("text", "").strip() for s in segs if isinstance(s, dict)).strip()
    return ""

In [3]:
def audio_files():
    return sorted(p for p in INPUT_DIR.iterdir() if p.suffix.lower() in AUDIO_EXTS)


def process(path):
    t0 = time.monotonic()
    job_id = submit(path)
    print(f"  submitted {path.name} -> job {job_id}")
    js = wait(job_id, path.name[:30])
    text = extract_text(js)
    rec = {"audio": path.name, "job_id": job_id, "text": text, "response": js}
    # write atomically so a half-written .json is never mistaken for a finished file
    tmp = OUTPUT_DIR / f"{path.stem}.json.tmp"
    tmp.write_text(json.dumps(rec, ensure_ascii=False, indent=1), encoding="utf-8")
    (OUTPUT_DIR / f"{path.stem}.txt").write_text(text + "\n", encoding="utf-8")
    os.replace(tmp, OUTPUT_DIR / f"{path.stem}.json")
    return path.name, len(text), time.monotonic() - t0


def run(limit=0, only=None):
    """Transcribe every file in INPUT_DIR that has no .json yet. limit: first N files (testing). only: substring filter."""
    status, health = http_json(f"{API}/health", timeout=10)
    if status != 200 or health.get("status") != "ready":
        raise RuntimeError(f"API not ready: {health}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    files = [f for f in audio_files() if not only or only in f.name]
    todo = [f for f in files if not (OUTPUT_DIR / f"{f.stem}.json").exists()]
    if limit:
        todo = todo[:limit]
    print(f"{len(files)} files, {len(files) - len(todo) if not limit else '-'} already done, {len(todo)} to do")

    failed = []
    t0 = time.monotonic()
    with ThreadPoolExecutor(WORKERS) as pool:
        futs = {pool.submit(process, f): f for f in todo}
        for i, fut in enumerate(as_completed(futs), 1):
            f = futs[fut]
            try:
                name, n_chars, secs = fut.result()
                print(f"[{i}/{len(todo)}] ok   {name}  ({n_chars} chars, {secs:.0f}s)")
            except Exception as e:
                failed.append(f.name)
                print(f"[{i}/{len(todo)}] FAIL {f.name}: {e}", file=sys.stderr)
    print(f"Done in {time.monotonic() - t0:.0f}s. Transcripts in {OUTPUT_DIR}"
          + (f"\n{len(failed)} failed (re-run to retry): {failed}" if failed else ""))

## Smoke test
Transcribes the smallest file (`abuser.wav`, ~12 MB). Check the text and the raw response shape before the full run.

In [4]:
print(http_json(f"{API}/health")[1])
files = audio_files()
print(len(files), "audio files in", INPUT_DIR)
run(only="abuser")

{'status': 'ready', 'device': 'cuda', 'device_target': 'cuda', 'fallback_reason': None, 'model_path': '/models/ft_model', 'default_language': 'auto', 'lid_bias': {}, 'supported_languages': ['uz', 'kk', 'ru'], 'queue_size': 0, 'num_workers': 4}
31 audio files in /Users/macbook/Documents/IT/projects/tts/data/raw_audios
1 files, 0 already done, 1 to do
  submitted abuser.wav -> job cee9fc4e-0618-4ead-bde1-b3b520e43104
  [abuser.wav] processing  server queue=0  waited 0s
  [abuser.wav] completed  server queue=0  waited 10s
[1/1] ok   abuser.wav  (1391 chars, 11s)
Done in 11s. Transcripts in /Users/macbook/Documents/IT/projects/tts/data/transcripts


In [ ]:
rec = json.loads((OUTPUT_DIR / "abuser.json").read_text(encoding="utf-8"))
print(rec["text"][:1000]); print(json.dumps(rec["response"], ensure_ascii=False, indent=1)[:1500])

## Full run
Long files take a while. Safe to interrupt and re-run — finished files are skipped. (An interrupted in-flight job may keep running on the server and be re-submitted.)

In [6]:
run()

31 files, 1 already done, 30 to do
  submitted 2026-yilda Google'ga ishga kirish XARITASI.wav -> job ad5aaf50-0209-4650-bff5-68a7a3e0a430
  [2026-yilda Google'ga ishga kir] processing  server queue=0  waited 0s
  submitted 28-son - GTA VI, Fransiyadagi noroziliklar, Tesla uchar avtomobili va boshqa yangiliklar tahlili.wav -> job 6ff2847d-3b89-4859-ab45-d86bd069ed6f
  [28-son - GTA VI, Fransiyadagi ] processing  server queue=0  waited 0s
  submitted 29-son - Krinjdan tortib siyosatgacha hafta davomida sodir bo‘lgan voqealar muhokamasi.wav -> job 2221823b-596e-45d7-8fc8-9b62066d1063
  [29-son - Krinjdan tortib siyos] processing  server queue=0  waited 0s
  submitted 40 daqiqada yillar davomida yo'qotganimizni qaytaramiz - Maqolim 1-son.wav -> job 6ddd7649-fe3c-4d0b-84dc-48491a4d391f
  [40 daqiqada yillar davomida yo] processing  server queue=0  waited 0s
  submitted 31 - son - Birinçi besoqol, musulmon, tub amerikalik senatorlar. Amerika siyosati kök rangda.wav -> job 10955b76-cc1c-4902-